In [18]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

from sklearn.svm import SVC

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_val_score
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

import pickle


# ==========================================
# LOAD DATASET
# ==========================================

df = pd.read_csv(
    "social_media_screentime_mental_health_2026.csv"
)


# ==========================================
# REMOVE ID COLUMN
# ==========================================

df = df.drop(
    "participant_id",
    axis=1
)


# ==========================================
# SEPARATE FEATURES AND TARGET
# ==========================================

X = df.drop(
    "wellbeing_band",
    axis=1
)

y = df[
    "wellbeing_band"
]


# ==========================================
# TRAIN TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)


# ==========================================
# IDENTIFY FEATURES
# ==========================================

numeric_features = [

    col

    for col in X.columns

    if X[col].dtype != "object"

]


categorical_features = [

    col

    for col in X.columns

    if X[col].dtype == "object"

]


# ==========================================
# NUMERICAL PIPELINE
# ==========================================

numeric_pipeline = Pipeline([

    (
        "imputer",

        SimpleImputer(
            strategy="mean"
        )
    ),

    (
        "scaler",

        StandardScaler()
    )

])


# ==========================================
# CATEGORICAL PIPELINE
# ==========================================

categorical_pipeline = Pipeline([

    (
        "imputer",

        SimpleImputer(
            strategy="most_frequent"
        )
    ),

    (
        "encoder",

        OneHotEncoder(
            handle_unknown="ignore"
        )
    )

])


# ==========================================
# PREPROCESSOR
# ==========================================

preprocessor = ColumnTransformer([

    (
        "numerical",

        numeric_pipeline,

        numeric_features
    ),

    (
        "categorical",

        categorical_pipeline,

        categorical_features
    )

])


# ==========================================
# COMPLETE PIPELINE
# ==========================================

pipeline = Pipeline([

    (
        "preprocessor",

        preprocessor
    ),

    (
        "classifier",

        SVC(
            probability=True
        )
    )

])


# ==========================================
# HYPERPARAMETER GRID
# ==========================================

param_grid = {

    "classifier__kernel": [

        "rbf",

        "linear",

        "poly"

    ],

    "classifier__C": [

        0.01,

        0.1,

        1,

        10,

        100

    ],

    "classifier__gamma": [

        "scale",

        "auto"

    ],

    "classifier__degree": [

        2,

        3,

        4

    ]

}


# ==========================================
# CROSS VALIDATION
# ==========================================

cv = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)


# ==========================================
# GRID SEARCH
# ==========================================

grid = GridSearchCV(

    estimator=pipeline,

    param_grid=param_grid,

    cv=cv,

    scoring="accuracy",

    n_jobs=-1

)


grid.fit(

    X_train,

    y_train

)


# ==========================================
# BEST PARAMETERS
# ==========================================

print(

    "Best Parameters:",

    grid.best_params_

)


print(

    "Best CV Score:",

    grid.best_score_

)


# ==========================================
# BEST MODEL
# ==========================================

model = grid.best_estimator_


# ==========================================
# TEST PREDICTION
# ==========================================

y_pred = model.predict(

    X_test

)


# ==========================================
# ACCURACY
# ==========================================

accuracy = accuracy_score(

    y_test,

    y_pred

)


print(

    f"Test Accuracy: "
    f"{accuracy * 100:.2f}%"

)


# ==========================================
# CONFUSION MATRIX
# ==========================================

print(

    confusion_matrix(

        y_test,

        y_pred

    )

)


# ==========================================
# CLASSIFICATION REPORT
# ==========================================

print(

    classification_report(

        y_test,

        y_pred

    )

)


# ==========================================
# CROSS-VALIDATION SCORES
# ==========================================

scores = cross_val_score(

    model,

    X_train,

    y_train,

    cv=cv,

    scoring="accuracy"

)


print(

    "Individual CV Scores:",

    scores

)


print(

    f"Mean CV Score: "
    f"{scores.mean() * 100:.2f}%"

)


print(

    f"Standard Deviation: "
    f"{scores.std() * 100:.2f}%"

)


# ==========================================
# SAVE COMPLETE PIPELINE
# ==========================================



Best Parameters: {'classifier__C': 100, 'classifier__degree': 2, 'classifier__gamma': 'auto', 'classifier__kernel': 'rbf'}
Best CV Score: 0.9528571428571428
Test Accuracy: 95.14%
[[185   0  10]
 [  0 432  20]
 [  3  35 715]]
              precision    recall  f1-score   support

     At-risk       0.98      0.95      0.97       195
        Good       0.93      0.96      0.94       452
    Moderate       0.96      0.95      0.95       753

    accuracy                           0.95      1400
   macro avg       0.96      0.95      0.95      1400
weighted avg       0.95      0.95      0.95      1400

Individual CV Scores: [0.95178571 0.95178571 0.96339286 0.94910714 0.94821429]
Mean CV Score: 95.29%
Standard Deviation: 0.55%


In [22]:
import pickle

with open("model.pkl", "wb") as file:
    pickle.dump(model, file)